[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sumilee-pcu/llm-textbook/blob/main/chapters/part4_advanced_ops/ch13_capstone.ipynb)

위 배지를 누르면 이 노트북이 구글 코랩에서 바로 열립니다.

# 13장 종합 프로젝트: 지식을 가진 에이전트
**「LLM 애플리케이션 입문 — RAG에서 멀티에이전트까지」 실습 노트북**

> 제4부 심화·운영·종합
>
> Tier · `[T1]` 코랩 즉시 실행 · `[T2]` GPU/장시간 · `[T3]` 이론(개념 점검)
>
> 실습 코드 저장소: github.com/sumilee-pcu/llm-textbook

## 환경 설정 · 한글 폰트와 라이브러리
코랩에서 처음 한 번만 실행합니다.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
# 한글 폰트 설치 + matplotlib 한글 적용 (코랩 최초 1회)
!apt-get -qq -y install fonts-nanum > /dev/null 2>&1
!rm -rf ~/.cache/matplotlib
import matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm.fontManager.__init__()
for f in fm.fontManager.ttflist:
    if "NanumGothic" in f.name:
        plt.rcParams["font.family"] = fm.FontProperties(fname=f.fname).get_name(); break
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
# 실습 라이브러리 설치 (검증 시점 2026-06, 하한 고정)
!pip install -q "google-genai>=1.0" "networkx>=3.0"

In [ ]:
# 설치된 핵심 라이브러리 버전 기록 (재현용) — 이 출력의 버전을 그대로 고정하면 가장 안정적이다
import importlib.metadata as _m
for _p in ["google-genai","langchain","langchain-google-genai","langchain-community",
           "langchain-text-splitters","langgraph","faiss-cpu","tiktoken","rank-bm25",
           "networkx","tavily-python"]:
    try: print(f"{_p}=={_m.version(_p)}")
    except Exception: pass

### API 키와 모델 설정

In [ ]:
# API 키 — 코랩 보안 비밀(시크릿)에 GOOGLE_API_KEY 등록 후 사용
#   왼쪽 열쇠 아이콘 > 새 보안 비밀 > 이름 GOOGLE_API_KEY > 값에 키 붙여넣기 > 노트북 액세스 ON
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)
GEMINI_FLASH = "gemini-2.5-flash"   # 모델 갱신 시 이 한 줄만 변경
EMBED_MODEL = "text-embedding-004"

### 검색 계층 헬퍼(embed·cosine)

In [ ]:
import numpy as np

def cosine(a, b):
    a, b = np.array(a), np.array(b)
    return a @ b / (np.linalg.norm(a) * np.linalg.norm(b))

def embed(text):
    res = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return res.embeddings[0].values

## 예제 13-1. 질문 유형 라우터

In [ ]:
def route(question):
    relation_words = ["연결", "관계", "통해", "모두", "어느 도시", "어디"]
    return "graph" if any(w in question for w in relation_words) else "vector"

for q in ["환불 며칠 안에 돼?", "이수미가 있는 곳은 어느 도시야?"]:
    print(route(q), "<-", q)

## 예제 13-2. 통합 지식 에이전트

In [ ]:
def knowledge_agent(question):
    if route(question) == "graph":
        evidence = hop("이수미")                       # 7장: 그래프 추적
    else:
        evidence = "\n".join(retrieve(question, k=2))   # 5장: 벡터 검색
    prompt = f"근거:\n{evidence}\n질문: {question}\n근거로만 답하라."
    return client.models.generate_content(model=GEMINI_FLASH, contents=prompt).text

print(knowledge_agent("환불 며칠 안에 돼?"))
print(knowledge_agent("이수미가 있는 곳은 어느 도시야?"))

## 심화 실습 `[T1]`
본문의 심화 실습 과제를 코랩에서 직접 구현해 본다. 책의 해당 장 끝 안내를 따른다.

## 정리
- 이 장의 예제를 모두 실행해 결과를 확인했다.
- 코드는 책 본문의 핵심을 옮긴 것이며, 확장 과제는 저장소 README를 참고한다.

> 저장소: github.com/sumilee-pcu/llm-textbook · 13장 노트북